In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fake_job_postings.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Dataset shape: (17880, 18)
Columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [2]:
TARGET = "fraudulent"

print(df[TARGET].value_counts(dropna=False))

fraudulent
0    17014
1      866
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

TARGET = "fraudulent"

# Step 1: Keep rows with valid target labels
df = df.loc[df[TARGET].isin([0, 1])].copy()

# Step 2: Remove the identifier
df = df.drop(columns=["job_id"])

# Step 3: Check duplicates after removing the identifier
print("Duplicates found:", df.duplicated().sum())

# Step 4: Remove exact duplicates if approved by your team
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)

# Step 5: Separate features and target
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Step 6: Create training and temporary sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Step 7: Divide the temporary set equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("\nFinal split sizes:")
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Duplicates found: 281
Dataset shape after cleaning: (17599, 17)

Final split sizes:
Training: (12319, 16) (12319,)
Validation: (2640, 16) (2640,)
Test: (2640, 16) (2640,)


In [4]:
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print("Class counts:")
    print(labels.value_counts())

    print("\nClass percentages:")
    print(
        (labels.value_counts(normalize=True) * 100).round(2)
    )


Training
Class counts:
fraudulent
0    11720
1      599
Name: count, dtype: int64

Class percentages:
fraudulent
0    95.14
1     4.86
Name: proportion, dtype: float64

Validation
Class counts:
fraudulent
0    2511
1     129
Name: count, dtype: int64

Class percentages:
fraudulent
0    95.11
1     4.89
Name: proportion, dtype: float64

Test
Class counts:
fraudulent
0    2512
1     128
Name: count, dtype: int64

Class percentages:
fraudulent
0    95.15
1     4.85
Name: proportion, dtype: float64


In [5]:
# Inspect the training feature columns
print(X_train.columns.tolist())

# Check data types
print(X_train.dtypes)

# Count missing values in training data
print(X_train.isnull().sum().sort_values(ascending=False))

# Count unique values in categorical columns
categorical_cols = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

for col in categorical_cols:
    if col in X_train.columns:
        print(f"{col}: {X_train[col].nunique(dropna=True)} unique values")

['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']
title                    str
location                 str
department               str
salary_range             str
company_profile          str
description              str
requirements             str
benefits                 str
telecommuting          int64
has_company_logo       int64
has_questions          int64
employment_type          str
required_experience      str
required_education       str
industry                 str
function                 str
dtype: object
salary_range           10346
department              7921
required_education      5584
benefits                4983
required_experience     4862
function                4461
industry                3341
employment_type         2391
company_profile         2290
requi

In [6]:
# Missing-value count and percentage
missing_summary = pd.DataFrame({
    "missing_count": X_train.isnull().sum(),
    "missing_percentage": (
        X_train.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

print(missing_summary)

                     missing_count  missing_percentage
salary_range                 10346               83.98
department                    7921               64.30
required_education            5584               45.33
benefits                      4983               40.45
required_experience           4862               39.47
function                      4461               36.21
industry                      3341               27.12
employment_type               2391               19.41
company_profile               2290               18.59
requirements                  1845               14.98
location                       247                2.01
title                            0                0.00
description                      0                0.00
has_questions                    0                0.00
telecommuting                    0                0.00
has_company_logo                 0                0.00


In [7]:
print("Salary examples:")
print(X_train["salary_range"].dropna().head(20).to_list())

print("\nMost common salary values:")
print(X_train["salary_range"].value_counts().head(15))

Salary examples:
['18000-20000', '40000-55000', '20000-50000', '70000-90000', '26000-52000', '30000-35000', '80000-250000', '13000-15000', '50000-72000', '45000-55000', '30000-50000', '80000-100000', '40000-60000', '90000-100000', '26000-28000', '55000-75000', '15000-30000', '100000-120000', '20000-24000', '1000-4000']

Most common salary values:
salary_range
0-0             103
40000-50000      47
30000-40000      37
25000-30000      26
80000-100000     25
45000-67000      25
30000-50000      21
55000-75000      21
50000-70000      21
70000-90000      20
40000-60000      19
40000-45000      19
35000-45000      19
30000-35000      17
60000-80000      17
Name: count, dtype: int64


In [8]:
import pandas as pd

def engineer_salary_features(data):
    data = data.copy()

    salary = data["salary_range"].astype("string").str.strip()

    # Record whether the original salary field was missing
    data["salary_missing"] = salary.isna().astype("int64")

    # Extract values only when the entire string is a numeric range
    extracted = salary.str.extract(
        r"^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$"
    )

    data["salary_min"] = pd.to_numeric(
        extracted[0], errors="coerce"
    )
    data["salary_max"] = pd.to_numeric(
        extracted[1], errors="coerce"
    )

    # Remove the original text column from model inputs
    data = data.drop(columns=["salary_range"])

    return data


X_train_salary = engineer_salary_features(X_train)

print(
    X_train_salary[
        ["salary_missing", "salary_min", "salary_max"]
    ].head(10)
)

print("\nNew feature missing values:")
print(
    X_train_salary[
        ["salary_missing", "salary_min", "salary_max"]
    ].isna().sum()
)

       salary_missing  salary_min  salary_max
12344               1        <NA>        <NA>
8952                1        <NA>        <NA>
14465               1        <NA>        <NA>
15875               0       18000       20000
5217                0       40000       55000
12984               1        <NA>        <NA>
2600                1        <NA>        <NA>
11336               1        <NA>        <NA>
8822                0       20000       50000
10234               1        <NA>        <NA>

New feature missing values:
salary_missing        0
salary_min        10365
salary_max        10365
dtype: int64


In [9]:
salary = X_train["salary_range"].astype("string").str.strip()

extracted = salary.str.extract(
    r"^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$"
)

unmatched = salary[
    salary.notna() & extracted[0].isna()
]

print("Unmatched non-missing salary entries:", len(unmatched))
print(unmatched.value_counts().head(30))

Unmatched non-missing salary entries: 19
salary_range
10-Oct    4
10-Nov    3
11-Nov    2
9-Dec     1
3-Apr     1
4-Apr     1
2-Apr     1
4-Jun     1
Jun-18    1
11-Dec    1
8-Sep     1
Oct-20    1
Dec-25    1
Name: count, dtype: Int64


In [10]:
import pandas as pd

def engineer_salary_features(data):
    data = data.copy()

    salary = data["salary_range"].astype("string").str.strip()

    # 1. Salary was genuinely missing
    data["salary_missing"] = salary.isna().astype("int64")

    # 2. Extract numeric ranges
    extracted = salary.str.extract(
        r"^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$"
    )

    data["salary_min"] = pd.to_numeric(
        extracted[0], errors="coerce"
    )
    data["salary_max"] = pd.to_numeric(
        extracted[1], errors="coerce"
    )

    # 3. Value existed but could not be parsed
    data["salary_unparsed"] = (
        salary.notna() & extracted[0].isna()
    ).astype("int64")

    # 4. Remove original salary text from model inputs
    data = data.drop(columns=["salary_range"])

    return data

In [11]:
X_train_prepared = engineer_salary_features(X_train)
X_val_prepared = engineer_salary_features(X_val)
X_test_prepared = engineer_salary_features(X_test)

In [12]:
print(
    X_train_prepared[
        [
            "salary_missing",
            "salary_unparsed",
            "salary_min",
            "salary_max"
        ]
    ].head(10)
)

print("\nSalary feature summary:")
print(
    X_train_prepared[
        [
            "salary_missing",
            "salary_unparsed",
            "salary_min",
            "salary_max"
        ]
    ].isna().sum()
)

print("\nUnparsed values:")
print(
    X_train.loc[
        X_train["salary_range"].notna()
        & X_train["salary_range"].astype("string").str.strip().str.extract(
            r"^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$"
        )[0].isna(),
        "salary_range"
    ].value_counts()
)

       salary_missing  salary_unparsed  salary_min  salary_max
12344               1                0        <NA>        <NA>
8952                1                0        <NA>        <NA>
14465               1                0        <NA>        <NA>
15875               0                0       18000       20000
5217                0                0       40000       55000
12984               1                0        <NA>        <NA>
2600                1                0        <NA>        <NA>
11336               1                0        <NA>        <NA>
8822                0                0       20000       50000
10234               1                0        <NA>        <NA>

Salary feature summary:
salary_missing         0
salary_unparsed        0
salary_min         10365
salary_max         10365
dtype: int64

Unparsed values:
salary_range
10-Oct    4
10-Nov    3
11-Nov    2
9-Dec     1
3-Apr     1
4-Apr     1
2-Apr     1
4-Jun     1
Jun-18    1
11-Dec    1
8-Sep     1
Oct-20

In [13]:
text_features = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

categorical_features = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

binary_features = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

salary_features = [
    "salary_min",
    "salary_max",
    "salary_missing",
    "salary_unparsed"
]

In [14]:
X_train_prepared = engineer_salary_features(X_train)
X_val_prepared = engineer_salary_features(X_val)
X_test_prepared = engineer_salary_features(X_test)

In [15]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Process text columns individually
text_transformers = []

for col in text_features:
    text_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value=""
        )),
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2)
        ))
    ])

    text_transformers.append(
        (f"tfidf_{col}", text_pipeline, col)
    )

# Process categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Process salary features
salary_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=False
    )),
    ("scaler", StandardScaler())
])

# Combine all transformations
preprocessor = ColumnTransformer(
    transformers=[
        *text_transformers,
        ("categorical", categorical_pipeline, categorical_features),
        ("salary", salary_pipeline, salary_features),
        ("binary", "passthrough", binary_features)
    ],
    remainder="drop"
)

In [16]:
import numpy as np
import pandas as pd

text_features = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

categorical_features = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

binary_features = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

salary_features = [
    "salary_min",
    "salary_max",
    "salary_missing",
    "salary_unparsed"
]


def prepare_input_types(data):
    data = data.copy()

    # Text columns: fill missing values and ensure strings
    for col in text_features:
        data[col] = data[col].fillna("").astype(str)

    # Categorical columns: fill missing values
    for col in categorical_features:
        data[col] = (
            data[col]
            .astype(object)
            .where(data[col].notna(), "Unknown")
        )

    # Salary columns: convert nullable pandas types to numeric
    for col in salary_features:
        data[col] = pd.to_numeric(
            data[col], errors="coerce"
        ).astype("float64")

    # Binary columns: ensure numeric values
    for col in binary_features:
        data[col] = pd.to_numeric(
            data[col], errors="raise"
        )

    return data


X_train_ready = prepare_input_types(X_train_prepared)
X_val_ready = prepare_input_types(X_val_prepared)
X_test_ready = prepare_input_types(X_test_prepared)

print(X_train_ready.dtypes)

title                      str
location                object
department              object
company_profile            str
description                str
requirements               str
benefits                   str
telecommuting            int64
has_company_logo         int64
has_questions            int64
employment_type         object
required_experience     object
required_education      object
industry                object
function                object
salary_missing         float64
salary_min             float64
salary_max             float64
salary_unparsed        float64
dtype: object


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Each text column receives its own TF-IDF vectorizer
text_transformers = [
    (
        f"tfidf_{col}",
        TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2)
        ),
        col
    )
    for col in text_features
]

# Categorical features
categorical_pipeline = Pipeline([
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

# Numeric salary features
salary_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        *text_transformers,
        ("categorical", categorical_pipeline, categorical_features),
        ("salary", salary_pipeline, salary_features),
        ("binary", "passthrough", binary_features)
    ],
    remainder="drop"
)

In [18]:
X_train_transformed = preprocessor.fit_transform(X_train_ready)

X_val_transformed = preprocessor.transform(X_val_ready)

X_test_transformed = preprocessor.transform(X_test_ready)

print("Training matrix:", X_train_transformed.shape)
print("Validation matrix:", X_val_transformed.shape)
print("Test matrix:", X_test_transformed.shape)

Training matrix: (12319, 53840)
Validation matrix: (2640, 53840)
Test matrix: (2640, 53840)
